# 01 · Statistical tests: delivery, vouchers and repeat purchase

**Question.** The SQL modules show patterns: late orders get worse reviews (a05), late first orders
seem to repeat less and voucher first orders seem to repeat more (a12). This notebook checks which
of those patterns are **statistically real**, how big they are, and how to test a fix properly.

| # | Question | Unit of analysis | Method |
|---|---|---|---|
| T1 | Are late deliveries more likely to get low reviews? | valid delivered order with a review | two-proportion z-test, Newcombe CI, relative risk |
| T2 | Does the review distribution differ across delay buckets? | same | chi-square test of independence, Cramér's V |
| T3 | Does a late **first** order reduce 180-day repeat? | repeat-eligible customer | two-proportion z-test + CI |
| T4 | Do voucher-paid first orders repeat more? | repeat-eligible customer | two-proportion z-test + CI |
| T5 | Is the late-first-order effect robust to confounders? | repeat-eligible customer | logistic regression, robust (HC1) SE, odds ratios |
| T6 | Is delivery slower in the North / Northeast? | valid delivery | bootstrap 95% CI of the median (2,000 resamples, seed 42) |
| T7 | How would we A/B test a second-order voucher? | design | power analysis (two proportions) |

**Conventions.** α = 0.05, two-sided. Differences are *group A − group B* in percentage points (pp).
Data come from MySQL views (`v_valid_orders`) and from
`results/sql/a12_repeat_and_time_to_second__customer_level_for_stats.csv`.
Outputs: `results/stats/tests.csv`, `results/stats/stats_summary.md` (+ detail CSVs) and the charts.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (confint_proportions_2indep, proportion_effectsize,
                                          proportions_ztest)

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":            # nbconvert runs inside notebooks/
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "python"))
from db import get_engine               # noqa: E402

engine = get_engine()
STATS_DIR = REPO / "results" / "stats"
STATS_DIR.mkdir(parents=True, exist_ok=True)
ALPHA = 0.05
rows = []        # one dict per test -> tests.csv
report = []      # markdown blocks  -> stats_summary.md

with engine.connect() as conn:
    cfg = dict(conn.exec_driver_sql("SELECT param_name, param_value FROM cfg_params").fetchall())
LOW_MAX = int(cfg["low_review_max"])
print("config:", cfg)

config: {'low_review_max': '2', 'min_orders_lane': '100', 'min_orders_seller': '30', 'repeat_horizon_days': '180', 'window_end': '2018-08-31', 'window_start': '2017-01-01'}


In [2]:
def pct(x, d=2):
    return f"{100 * x:.{d}f}%"


def fmt_p(p):
    # p-values below 1e-4 are printed as "< 0.0001" (they underflow to 0 in float)
    return "< 0.0001" if p < 1e-4 else f"{p:.3g}"


def two_proportion_test(test_id, question, hypothesis, unit, label_a, k_a, n_a, label_b, k_b, n_b,
                        plain_english, caveat):
    # Two-proportion z-test (A vs B), Newcombe 95% CI of the difference, relative risk, Cohen's h.
    p_a, p_b = k_a / n_a, k_b / n_b
    z, p_value = proportions_ztest([k_a, k_b], [n_a, n_b], alternative="two-sided")
    ci_low, ci_high = confint_proportions_2indep(k_a, n_a, k_b, n_b, method="newcomb",
                                                 compare="diff", alpha=ALPHA)
    rr = p_a / p_b
    h = proportion_effectsize(p_a, p_b)                       # Cohen's h
    significant = p_value < ALPHA
    verdict = "significant" if significant else "NOT significant"
    conclusion = (f"{label_a} {pct(p_a)} vs {label_b} {pct(p_b)}: difference {100*(p_a-p_b):+.2f} pp "
                  f"(95% CI {100*ci_low:+.2f} to {100*ci_high:+.2f} pp), relative risk {rr:.2f}; "
                  f"{verdict} at alpha = {ALPHA} (p {'= ' + fmt_p(p_value) if p_value >= 1e-4 else fmt_p(p_value)}). {plain_english}")
    block = "\n".join([
        f"### {test_id}. {question}",
        f"- **Hypothesis:** {hypothesis}",
        f"- **Unit of analysis:** {unit}",
        f"- **n per group:** {label_a} = {n_a:,} · {label_b} = {n_b:,}",
        f"- **Rates:** {label_a} = {pct(p_a)} ({k_a:,}/{n_a:,}) · {label_b} = {pct(p_b)} ({k_b:,}/{n_b:,})",
        f"- **Difference (A − B):** {100*(p_a-p_b):+.2f} pp, 95% CI [{100*ci_low:+.2f}, {100*ci_high:+.2f}] pp (Newcombe)",
        f"- **Test statistic:** z = {z:.2f} · **p-value:** {fmt_p(p_value)}",
        f"- **Effect size:** relative risk = {rr:.2f} · Cohen's h = {h:.3f}",
        f"- **Conclusion:** {conclusion}",
        f"- **Caveat:** {caveat}",
    ])
    print(block.replace("**", ""))
    report.append(block)
    rows.append({"test_id": test_id, "question": question, "unit": unit,
                 "method": "two-proportion z-test; Newcombe CI",
                 "group_a": label_a, "n_a": n_a, "rate_a": round(p_a, 6),
                 "group_b": label_b, "n_b": n_b, "rate_b": round(p_b, 6),
                 "effect": round(p_a - p_b, 6), "effect_type": "difference in proportions (A-B)",
                 "ci_low": round(ci_low, 6), "ci_high": round(ci_high, 6),
                 "statistic": round(z, 4), "p_value": p_value,
                 "effect_size": round(h, 4), "effect_size_type": "Cohen's h",
                 "significant": significant, "conclusion": conclusion, "caveat": caveat})

## T1 · Late delivery → low review?
Unit: each valid delivery (delivered, dated, no timestamp anomaly) that received a review.

In [3]:
with engine.connect() as conn:
    reviewed = pd.read_sql(
        "SELECT order_id, is_late, review_score, delay_bucket "
        "FROM v_valid_orders WHERE is_valid_delivery = 1 AND review_score IS NOT NULL", conn)
reviewed["is_low"] = (reviewed["review_score"] <= LOW_MAX).astype(int)
late = reviewed[reviewed["is_late"] == 1]
ontime = reviewed[reviewed["is_late"] == 0]

two_proportion_test(
    "T1", "Are late deliveries more likely to get a low (1-2 star) review?",
    "H0: P(low review | late) = P(low review | on time); H1: they differ.",
    "valid delivered order with a review",
    "late orders", int(late["is_low"].sum()), len(late),
    "on-time orders", int(ontime["is_low"].sum()), len(ontime),
    plain_english="Lateness is the strongest driver of 1-2 star reviews in this data.",
    caveat="Observational: late orders also differ in route and product mix, but the size of the gap "
           "and the dose-response in T2 make a causal link very plausible.")

### T1. Are late deliveries more likely to get a low (1-2 star) review?
- Hypothesis: H0: P(low review | late) = P(low review | on time); H1: they differ.
- Unit of analysis: valid delivered order with a review
- n per group: late orders = 6,354 · on-time orders = 87,844
- Rates: late orders = 62.48% (3,970/6,354) · on-time orders = 9.24% (8,118/87,844)
- Difference (A − B): +53.24 pp, 95% CI [+52.03, +54.44] pp (Newcombe)
- Test statistic: z = 122.53 · p-value: < 0.0001
- Effect size: relative risk = 6.76 · Cohen's h = 1.205
- Conclusion: late orders 62.48% vs on-time orders 9.24%: difference +53.24 pp (95% CI +52.03 to +54.44 pp), relative risk 6.76; significant at alpha = 0.05 (p < 0.0001). Lateness is the strongest driver of 1-2 star reviews in this data.
- Caveat: Observational: late orders also differ in route and product mix, but the size of the gap and the dose-response in T2 make a causal link very plausible.


## T2 · Review distribution across delay buckets
Chi-square test of independence on the bucket × score table. Cramér's V measures the strength
(0 = none, 1 = perfect association).

In [4]:
table = pd.crosstab(reviewed["delay_bucket"], reviewed["review_score"])
chi2, p_chi, dof, expected = chi2_contingency(table)
n_total = table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n_total * (min(table.shape) - 1)))
row_pct = (100 * table.div(table.sum(axis=1), axis=0)).round(1)
print("Observed counts:\n", table, "\n\nRow % (share of each score within a bucket):\n", row_pct)
print(f"\nMinimum expected cell count = {expected.min():.1f} (chi-square needs >= 5)")

conclusion = (f"Score distribution depends strongly on delay (chi2 = {chi2:,.0f}, dof = {dof}, "
              f"p {fmt_p(p_chi)}, Cramér's V = {cramers_v:.3f}). 5-star share falls from "
              f"{row_pct.iloc[0][5]:.1f}% (7+ days early) to {row_pct.iloc[-1][5]:.1f}% (8+ days late); "
              f"1-star share rises from {row_pct.iloc[0][1]:.1f}% to {row_pct.iloc[-1][1]:.1f}%.")
caveat = "With n this large almost any association is 'significant'; Cramér's V and the row % show it is also large."
block = "\n".join([
    "### T2. Does the review-score distribution differ across delay buckets?",
    "- **Hypothesis:** H0: review score is independent of delay bucket; H1: not independent.",
    "- **Unit of analysis:** valid delivered order with a review",
    f"- **n:** {n_total:,} orders in {table.shape[0]} buckets x {table.shape[1]} scores",
    f"- **Test statistic:** chi2 = {chi2:,.1f}, dof = {dof} · **p-value:** {fmt_p(p_chi)}",
    f"- **Effect size:** Cramér's V = {cramers_v:.3f}",
    f"- **Conclusion:** {conclusion}",
    f"- **Caveat:** {caveat}",
])
print("\n" + block.replace("**", ""))
report.append(block)
table.assign(total=table.sum(axis=1)).to_csv(STATS_DIR / "t2_delay_bucket_by_score_counts.csv")
rows.append({"test_id": "T2", "question": "Does the review distribution differ across delay buckets?",
             "unit": "valid delivered order with a review", "method": "chi-square test of independence",
             "n_a": n_total, "effect": round(cramers_v, 4), "effect_type": "Cramér's V",
             "statistic": round(chi2, 2), "p_value": p_chi, "effect_size": round(cramers_v, 4),
             "effect_size_type": "Cramér's V", "significant": p_chi < ALPHA,
             "conclusion": conclusion, "caveat": caveat})

Observed counts:
 review_score      1     2     3      4      5
delay_bucket                                 
1. Early 7+ d  4798  1857  5657  14900  47045
2. On time      995   468  1467   3023   7634
3. Late 1-3 d   464   130   270    367    610
4. Late 4-7 d  1019   159   171    146    246
5. Late 8+ d   1935   263   249    130    195 

Row % (share of each score within a bucket):
 review_score      1    2     3     4     5
delay_bucket                              
1. Early 7+ d   6.5  2.5   7.6  20.1  63.4
2. On time      7.3  3.4  10.8  22.2  56.2
3. Late 1-3 d  25.2  7.1  14.7  19.9  33.1
4. Late 4-7 d  58.5  9.1   9.8   8.4  14.1
5. Late 8+ d   69.8  9.5   9.0   4.7   7.0

Minimum expected cell count = 53.2 (chi-square needs >= 5)

### T2. Does the review-score distribution differ across delay buckets?
- Hypothesis: H0: review score is independent of delay bucket; H1: not independent.
- Unit of analysis: valid delivered order with a review
- n: 94,198 orders in 5 buckets x 5 sc

## T3 & T4 · First-order experience → 180-day repeat
Unit: **repeat-eligible customers** (first order on/before window_end − 180 days, so everyone had 180
days to come back). Repeat = a valid order on a later calendar day within 180 days.

In [5]:
cust = pd.read_csv(REPO / "results" / "sql" / "a12_repeat_and_time_to_second__customer_level_for_stats.csv")
eligible = cust[cust["eligible_repeat"] == 1].copy()
print(f"customers: {len(cust):,} | repeat-eligible: {len(eligible):,} | "
      f"eligible repeat rate: {pct(eligible['repeat_180d'].mean())}")

delivered_first = eligible[eligible["first_late"].notna()]
l1 = delivered_first[delivered_first["first_late"] == 1]
l0 = delivered_first[delivered_first["first_late"] == 0]
two_proportion_test(
    "T3", "Does a late first order reduce 180-day repeat purchase?",
    "H0: repeat rate after a late first order = after an on-time first order; H1: they differ.",
    "repeat-eligible customer whose first order was a valid delivery",
    "late first order", int(l1["repeat_180d"].sum()), len(l1),
    "on-time first order", int(l0["repeat_180d"].sum()), len(l0),
    plain_english="Direction matches the hypothesis (late -> fewer returns), but with a ~2% base rate "
                  "and only ~3.9k late first orders the test cannot rule out zero effect.",
    caveat="Low base rate (about 2%) means low power: an effect of this size needs far more late "
           "first orders to be detected. Observational, so route/region/category confounding is "
           "possible (see T5).")

customers: 94,703 | repeat-eligible: 57,283 | eligible repeat rate: 1.99%
### T3. Does a late first order reduce 180-day repeat purchase?
- Hypothesis: H0: repeat rate after a late first order = after an on-time first order; H1: they differ.
- Unit of analysis: repeat-eligible customer whose first order was a valid delivery
- n per group: late first order = 3,918 · on-time first order = 52,108
- Rates: late first order = 1.58% (62/3,918) · on-time first order = 2.02% (1,054/52,108)
- Difference (A − B): -0.44 pp, 95% CI [-0.81, +0.02] pp (Newcombe)
- Test statistic: z = -1.90 · p-value: 0.0571
- Effect size: relative risk = 0.78 · Cohen's h = -0.033
- Conclusion: late first order 1.58% vs on-time first order 2.02%: difference -0.44 pp (95% CI -0.81 to +0.02 pp), relative risk 0.78; NOT significant at alpha = 0.05 (p = 0.0571). Direction matches the hypothesis (late -> fewer returns), but with a ~2% base rate and only ~3.9k late first orders the test cannot rule out zero effect.
- Cavea

In [6]:
v1 = eligible[eligible["first_voucher"] == 1]
v0 = eligible[eligible["first_voucher"] == 0]
two_proportion_test(
    "T4", "Do voucher-paid first orders repeat more within 180 days?",
    "H0: repeat rate with a voucher on the first order = without; H1: they differ.",
    "repeat-eligible customer",
    "voucher first order", int(v1["repeat_180d"].sum()), len(v1),
    "no voucher", int(v0["repeat_180d"].sum()), len(v0),
    plain_english="Voucher users look slightly more likely to return, but the evidence is weak.",
    caveat="Selection bias: customers who hunt for vouchers differ from those who do not (deal-seeking, "
           "prior relationship). Even a significant gap would not prove that giving vouchers creates "
           "repeat purchases - that needs an experiment (T7).")

### T4. Do voucher-paid first orders repeat more within 180 days?
- Hypothesis: H0: repeat rate with a voucher on the first order = without; H1: they differ.
- Unit of analysis: repeat-eligible customer
- n per group: voucher first order = 2,316 · no voucher = 54,967
- Rates: voucher first order = 2.50% (58/2,316) · no voucher = 1.97% (1,082/54,967)
- Difference (A − B): +0.54 pp, 95% CI [-0.04, +1.26] pp (Newcombe)
- Test statistic: z = 1.81 · p-value: 0.0705
- Effect size: relative risk = 1.27 · Cohen's h = 0.036
- Conclusion: voucher first order 2.50% vs no voucher 1.97%: difference +0.54 pp (95% CI -0.04 to +1.26 pp), relative risk 1.27; NOT significant at alpha = 0.05 (p = 0.0705). Voucher users look slightly more likely to return, but the evidence is weak.
- Caveat: Selection bias: customers who hunt for vouchers differ from those who do not (deal-seeking, prior relationship). Even a significant gap would not prove that giving vouchers creates repeat purchases - that needs an exp

## T5 · Logistic regression: is the late-first-order effect robust to confounders?
`repeat_180d ~ first_late + first_voucher + log(first_value) + region + first-category (top 10, rest = other)`,
robust (HC1) standard errors. **Review score is deliberately excluded**: it sits on the causal path
late delivery → bad review → no return (a *mediator*), so controlling for it would hide part of the
very effect we want to measure.

In [7]:
d5 = delivered_first.copy()
top10 = d5["first_category"].value_counts().head(10).index
d5["first_category_top10"] = np.where(d5["first_category"].isin(top10), d5["first_category"], "other")
formula = ("repeat_180d ~ first_late + first_voucher + np.log(first_value)"
           " + C(region, Treatment(reference='Southeast'))"
           " + C(first_category_top10, Treatment(reference='other'))")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    logit = smf.logit(formula, data=d5).fit(cov_type="HC1", disp=False)
ci = logit.conf_int(alpha=ALPHA)
or_table = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ci_low": np.exp(ci[0]),
                         "ci_high": np.exp(ci[1]), "p_value": logit.pvalues}).round(4)
or_table.to_csv(STATS_DIR / "t5_logit_odds_ratios.csv")
print(f"n = {int(logit.nobs):,} customers, events = {int(d5['repeat_180d'].sum()):,}, "
      f"pseudo R2 (McFadden) = {logit.prsquared:.4f}\n")
print(or_table.to_string())

orl = or_table.loc["first_late"]
orv = or_table.loc["first_voucher"]
orf = or_table.loc["np.log(first_value)"]
conclusion = (f"Holding voucher use, first-order value, region and category fixed, a late first order has "
              f"odds ratio {orl.odds_ratio:.2f} (95% CI {orl.ci_low:.2f}-{orl.ci_high:.2f}, p = {orl.p_value:.3g}) "
              f"for repeating within 180 days. Voucher: OR {orv.odds_ratio:.2f} "
              f"({orv.ci_low:.2f}-{orv.ci_high:.2f}, p = {orv.p_value:.3g}); "
              f"log first-order value: OR {orf.odds_ratio:.2f} ({orf.ci_low:.2f}-{orf.ci_high:.2f}, "
              f"p = {orf.p_value:.3g}).")
caveat = ("Controls only for observed confounders; pseudo R2 is tiny, i.e. repeat purchase is mostly driven "
          "by things not in the data (needs, competition, marketing). Review score excluded on purpose "
          "(mediator).")
block = "\n".join([
    "### T5. Is the late-first-order effect robust to confounders?",
    "- **Hypothesis:** H0: odds ratio of first_late = 1 after controls; H1: OR ≠ 1.",
    "- **Unit of analysis:** repeat-eligible customer whose first order was a valid delivery",
    f"- **n:** {int(logit.nobs):,} customers, {int(d5['repeat_180d'].sum()):,} repeaters",
    f"- **Model:** `{formula}`, robust HC1 SE, McFadden pseudo R2 = {logit.prsquared:.4f}",
    f"- **Key estimate:** first_late OR = {orl.odds_ratio:.2f}, 95% CI [{orl.ci_low:.2f}, {orl.ci_high:.2f}], p = {orl.p_value:.3g}",
    f"- **Conclusion:** {conclusion}",
    f"- **Caveat:** {caveat}",
    "- Full table: `results/stats/t5_logit_odds_ratios.csv`",
])
print("\n" + block.replace("**", ""))
report.append(block)
rows.append({"test_id": "T5", "question": "Is the late-first-order effect robust to confounders?",
             "unit": "repeat-eligible customer with a validly delivered first order",
             "method": "logistic regression, HC1 robust SE", "n_a": int(logit.nobs),
             "effect": orl.odds_ratio, "effect_type": "odds ratio (first_late)",
             "ci_low": orl.ci_low, "ci_high": orl.ci_high, "statistic": round(logit.tvalues["first_late"], 4),
             "p_value": orl.p_value, "effect_size": orl.odds_ratio, "effect_size_type": "odds ratio",
             "significant": orl.p_value < ALPHA, "conclusion": conclusion, "caveat": caveat})

n = 56,026 customers, events = 1,116, pseudo R2 (McFadden) = 0.0067

                                                                                odds_ratio  ci_low  ci_high  p_value
Intercept                                                                           0.0212  0.0145   0.0312   0.0000
C(region, Treatment(reference='Southeast'))[T.Center-West]                          0.9487  0.7371   1.2211   0.6828
C(region, Treatment(reference='Southeast'))[T.North]                                0.7520  0.4689   1.2063   0.2372
C(region, Treatment(reference='Southeast'))[T.Northeast]                            0.7032  0.5583   0.8858   0.0028
C(region, Treatment(reference='Southeast'))[T.South]                                0.7785  0.6489   0.9341   0.0071
C(first_category_top10, Treatment(reference='other'))[T.bed_bath_table]             1.4128  1.1666   1.7109   0.0004
C(first_category_top10, Treatment(reference='other'))[T.computers_accessories]      0.7791  0.5902   1.0285   0.

## T6 · Delivery time by customer region (bootstrap CI of the median)
Delivery days are skewed (long right tail), so the **median** is the fair "typical" value; the bootstrap
gives its uncertainty without assuming a distribution: resample each region's orders with replacement
2,000 times (seed 42) and take the 2.5th-97.5th percentile of the resampled medians.

In [8]:
with engine.connect() as conn:
    deliveries = pd.read_sql(
        "SELECT c.region AS customer_region, vo.delivery_days "
        "FROM v_valid_orders AS vo INNER JOIN customers AS c ON c.customer_id = vo.customer_id "
        "WHERE vo.is_valid_delivery = 1", conn)

B, rng = 2000, np.random.default_rng(42)
boot = {}
summary = []
for region, grp in deliveries.groupby("customer_region"):
    arr = grp["delivery_days"].to_numpy()
    meds = np.array([np.median(rng.choice(arr, size=arr.size, replace=True)) for _ in range(B)])
    boot[region] = meds
    lo, hi = np.percentile(meds, [2.5, 97.5])
    summary.append({"customer_region": region, "n_deliveries": arr.size, "median_days": np.median(arr),
                    "ci_low": lo, "ci_high": hi, "mean_days": round(arr.mean(), 2)})
t6 = pd.DataFrame(summary).sort_values("median_days", ascending=False)
for region in ["North", "Northeast"]:                       # difference vs Southeast (largest region)
    diff = boot[region] - boot["Southeast"]
    t6.loc[t6.customer_region == region, "diff_vs_southeast"] = (
        np.median(deliveries.loc[deliveries.customer_region == region, "delivery_days"])
        - np.median(deliveries.loc[deliveries.customer_region == "Southeast", "delivery_days"]))
    t6.loc[t6.customer_region == region, "diff_ci_low"] = np.percentile(diff, 2.5)
    t6.loc[t6.customer_region == region, "diff_ci_high"] = np.percentile(diff, 97.5)
t6.to_csv(STATS_DIR / "t6_bootstrap_median_delivery_days.csv", index=False)
print(t6.to_string(index=False))

for region in ["North", "Northeast"]:
    r = t6.set_index("customer_region").loc[region]
    se = t6.set_index("customer_region").loc["Southeast"]
    conclusion = (f"Median delivery in the {region} is {r.median_days:.0f} days (95% CI {r.ci_low:.0f}-{r.ci_high:.0f}) "
                  f"vs {se.median_days:.0f} days in the Southeast (CI {se.ci_low:.0f}-{se.ci_high:.0f}): "
                  f"{r.diff_vs_southeast:+.0f} days (bootstrap CI {r.diff_ci_low:+.0f} to {r.diff_ci_high:+.0f}). "
                  f"The CI excludes 0, so the {region} is reliably slower.")
    rows.append({"test_id": f"T6-{region}", "question": f"Is delivery slower in the {region} than the Southeast?",
                 "unit": "valid delivery", "method": "bootstrap 95% CI of the median (2,000 resamples, seed 42)",
                 "group_a": region, "n_a": int(r.n_deliveries), "rate_a": r.median_days,
                 "group_b": "Southeast", "n_b": int(se.n_deliveries), "rate_b": se.median_days,
                 "effect": r.diff_vs_southeast, "effect_type": "difference in median delivery days",
                 "ci_low": r.diff_ci_low, "ci_high": r.diff_ci_high,
                 "significant": not (r.diff_ci_low <= 0 <= r.diff_ci_high),
                 "conclusion": conclusion,
                 "caveat": "Medians of whole days move in 1-day steps, so CIs are coarse; distance, not "
                           "region per se, drives the gap (a20)."})
block = "\n".join(["### T6. Is delivery time worse in the North / Northeast?",
                   "- **Unit of analysis:** valid delivery (order)",
                   "- **Method:** bootstrap 95% CI of the median, 2,000 resamples, seed 42",
                   "", t6.to_markdown(index=False), "",
                   f"- **Conclusion:** {rows[-2]['conclusion']} {rows[-1]['conclusion']}",
                   f"- **Caveat:** {rows[-1]['caveat']}"])
report.append(block)

customer_region  n_deliveries  median_days  ci_low  ci_high  mean_days  diff_vs_southeast  diff_ci_low  diff_ci_high
          North          1771         20.0    20.0     21.0      22.60               11.0         11.0          12.0
      Northeast          8890         17.0    17.0     18.0      20.00                8.0          8.0           9.0
    Center-West          5515         13.0    13.0     14.0      15.03                NaN          NaN           NaN
          South         13550         12.0    12.0     12.0      14.05                NaN          NaN           NaN
      Southeast         65108          9.0     9.0      9.0      10.72                NaN          NaN           NaN


## T7 · Designing the "second-order voucher" A/B test (power analysis)
Baseline = observed 180-day repeat rate of eligible customers. Minimum detectable effect (MDE) =
+20% **relative**. α = 0.05 (two-sided), power = 0.80, 1:1 split.

In [9]:
p1 = eligible["repeat_180d"].mean()
p2 = p1 * 1.20
effect = proportion_effectsize(p2, p1)                     # Cohen's h
n_per_arm = NormalIndPower().solve_power(effect_size=effect, alpha=ALPHA, power=0.80, ratio=1.0,
                                         alternative="two-sided")
n_per_arm = int(np.ceil(n_per_arm))
monthly_new = cust.groupby("first_order_month").size()
avg_new_2018 = monthly_new[monthly_new.index.str.startswith("2018")].mean()
weeks_to_enrol = 2 * n_per_arm / (avg_new_2018 / (365.25 / 12 / 7))
follow_up_weeks = 180 / 7
conclusion = (f"Baseline {pct(p1)} -> target {pct(p2)} needs {n_per_arm:,} new customers per arm "
              f"({2*n_per_arm:,} total). At {avg_new_2018:,.0f} new customers/month (2018 average) that is "
              f"~{weeks_to_enrol:.0f} weeks of enrolment (with every new customer in the test), plus "
              f"{follow_up_weeks:.0f} weeks of follow-up to observe the 180-day outcome.")
guardrails = ("AOV of the second order, margin proxy (voucher cost as % of the second order's GMV "
              "and of total GMV), low-review %, cancellation rate; also track 90-day repeat as an early read.")
block = "\n".join([
    "### T7. How would we A/B test a second-order voucher?",
    f"- **Design:** randomise new customers at their first delivery into control vs voucher-for-the-next-order; "
    f"primary metric = repeat within 180 days.",
    f"- **Baseline:** {pct(p1)} (eligible customers, n = {len(eligible):,}); **MDE:** +20% relative -> {pct(p2)} "
    f"(+{100*(p2-p1):.2f} pp); Cohen's h = {effect:.4f}",
    f"- **Sample size:** {n_per_arm:,} per arm (alpha = 0.05, power = 0.80, two-sided)",
    f"- **Duration:** {conclusion}",
    f"- **Guardrail metrics:** {guardrails}",
    "- **Caveat:** a 20% lift on a 2% base is small in absolute terms; a test this long only pays off if "
    "the voucher's cost is below the margin of the extra orders. Consider a larger MDE or a cheaper proxy metric "
    "(90-day repeat) to shorten it.",
])
print(block.replace("**", ""))
report.append(block)
rows.append({"test_id": "T7", "question": "A/B test design for a second-order voucher",
             "unit": "new customer (randomised at first delivery)",
             "method": "power analysis: proportion_effectsize + NormalIndPower.solve_power",
             "group_a": "voucher arm (target)", "n_a": n_per_arm, "rate_a": round(p2, 6),
             "group_b": "control (baseline)", "n_b": n_per_arm, "rate_b": round(p1, 6),
             "effect": round(p2 - p1, 6), "effect_type": "MDE (absolute)",
             "effect_size": round(effect, 4), "effect_size_type": "Cohen's h",
             "conclusion": conclusion, "caveat": "Guardrails: " + guardrails})

### T7. How would we A/B test a second-order voucher?
- Design: randomise new customers at their first delivery into control vs voucher-for-the-next-order; primary metric = repeat within 180 days.
- Baseline: 1.99% (eligible customers, n = 57,283); MDE: +20% relative -> 2.39% (+0.40 pp); Cohen's h = 0.0272
- Sample size: 21,174 per arm (alpha = 0.05, power = 0.80, two-sided)
- Duration: Baseline 1.99% -> target 2.39% needs 21,174 new customers per arm (42,348 total). At 6,459 new customers/month (2018 average) that is ~29 weeks of enrolment (with every new customer in the test), plus 26 weeks of follow-up to observe the 180-day outcome.
- Guardrail metrics: AOV of the second order, margin proxy (voucher cost as % of the second order's GMV and of total GMV), low-review %, cancellation rate; also track 90-day repeat as an early read.
- Caveat: a 20% lift on a 2% base is small in absolute terms; a test this long only pays off if the voucher's cost is below the margin of the extra orders. 

## Save outputs and charts

In [10]:
tests = pd.DataFrame(rows)
tests.to_csv(STATS_DIR / "tests.csv", index=False)

summary_cols = ["test_id", "n_a", "n_b", "effect_type", "effect", "ci_low", "ci_high", "p_value", "significant"]
header = [
    "# Statistical tests - summary",
    "",
    "Generated by `notebooks/01_statistical_tests.ipynb` (`make stats`). alpha = 0.05, two-sided. "
    "Differences are group A minus group B. Source data: MySQL views and "
    "`results/sql/a12_repeat_and_time_to_second__customer_level_for_stats.csv`.",
    "",
    "| Test | n (A / B) | Effect | 95% CI | p-value | Conclusion |",
    "|---|---|---|---|---|---|",
]
def fmt(v, kind):
    if pd.isna(v):
        return "-"
    if kind.startswith("difference in proportions") or kind.startswith("MDE"):
        return f"{100*v:+.2f} pp"
    if kind.startswith("difference in median"):
        return f"{v:+.0f} days"
    return f"{v:.3f}"
for r in rows:
    kind = r.get("effect_type", "")
    n = f"{r.get('n_a', '-'):,}" + (f" / {r['n_b']:,}" if pd.notna(r.get("n_b", np.nan)) else "")
    ci = (f"[{fmt(r['ci_low'], kind)}, {fmt(r['ci_high'], kind)}]"
          if pd.notna(r.get("ci_low", np.nan)) else "-")
    p = fmt_p(r["p_value"]) if pd.notna(r.get("p_value", np.nan)) else "-"
    header.append(f"| {r['test_id']} | {n} | {kind}: {fmt(r['effect'], kind)} | {ci} | {p} | "
                  f"{r['conclusion'].split('. ')[0]}. |")
(STATS_DIR / "stats_summary.md").write_text("\n".join(header + ["", "## Details", ""] + report) + "\n")
print(f"wrote {STATS_DIR / 'tests.csv'} and stats_summary.md ({len(rows)} rows)")

import make_charts                                     # noqa: E402  (python/make_charts.py)
make_charts.main()

wrote /Users/truesalt/Downloads/SQL Project/results/stats/tests.csv and stats_summary.md (8 rows)


saved results/charts/01_monthly_gmv.png
saved results/charts/02_review_by_delay.png


saved results/charts/03_cohort_heatmap.png
saved results/charts/04_repeat_late_vs_ontime_ci.png


saved results/charts/05_lead_funnel.png
saved results/charts/06_pareto_sellers.png


saved results/charts/07_lane_late_heatmap.png


0